# M00 — Bước 3: Baseline Models

Notebook 02 tương ứng bước 3 của workflow; notebook 01 chứa bước 1 và bước 2.
So sánh Dummy Median, Linear Regression và Random Forest trên split đã cố định.
Chỉ đọc train (60%) và validation (15%); không đọc calibration/test.
MAE validation là metric chính; RMSE và R² bổ sung. MAE/RMSE dùng tỷ VND.

Quy ước: Cxx là Code, Mxx là Markdown; gọi bằng mã, không dùng số execution.
C01 imports → C02 đọc split → C03 X/y → C04 metrics → C05 Dummy → C06 pipeline → C07 Linear Regression → M08 nhận xét → C09 Random Forest → C10 so sánh → M11 kết luận.

Các output và nhận xét cũ được giữ từ lần chạy trước khi tách. Cell thiết lập mới chưa chạy; Restart Kernel rồi Run All để xác minh notebook độc lập.


In [ ]:
# C01 — Imports cho baseline
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# C02 — Đọc train/validation đã cố định; không chia lại dữ liệu
PROJECT_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "processed" / "splits_v1" / "train.csv").is_file()
     and (folder / "notebooks").is_dir()), None,
)
assert PROJECT_ROOT is not None, "Mở kernel trong repo hoặc notebooks."
SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits_v1"
SEED = 42
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
assert train_df["row_id"].is_unique and val_df["row_id"].is_unique
assert set(train_df["row_id"]).isdisjoint(val_df["row_id"])
print("Train:", train_df.shape, "Validation:", val_df.shape)


In [ ]:
# ===== C03 - Feature allowlist, tách X/y =====

TARGET = "Price"

FEATURES = [
    "Area",
    "Frontage",
    "Access Road",
    "House direction",
    "Balcony direction",
    "Floors",
    "Bedrooms",
    "Bathrooms",
    "Legal status",
    "Furniture state",
    "City",
    "District",
]

# Kiểm tra schema đúng như handoff
assert set(train_df.columns) == set(
    FEATURES + [TARGET, "row_id"]
), "Schema train_df khác handoff."

assert set(val_df.columns) == set(
    FEATURES + [TARGET, "row_id"]
), "Schema val_df khác handoff."

# Tách feature và target
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[FEATURES].copy()
y_val = val_df[TARGET].copy()

# Kiểm tra nhanh
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nFeatures:")
print(FEATURES)

X_train: (18136, 12)
y_train: (18136,)
X_val: (4535, 12)
y_val: (4535,)

Features:
['Area', 'Frontage', 'Access Road', 'House direction', 'Balcony direction', 'Floors', 'Bedrooms', 'Bathrooms', 'Legal status', 'Furniture state', 'City', 'District']


In [ ]:
# ===== C04 - Evaluation metrics (MAE, RMSE, R²) =====
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)

    rmse = mean_squared_error(
        y_true,
        y_pred
    ) ** 0.5

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }

In [ ]:
# ===== C05 - Dummy Median Baseline =====
import numpy as np

# Median của tập train
median_price = y_train.median()

# Dự đoán mọi căn nhà bằng cùng một giá
dummy_pred_val = np.full(
    len(y_val),
    median_price
)

# Tính metric
dummy_metrics = evaluate(
    y_val,
    dummy_pred_val
)

print("Median Price:", median_price)
print()

for metric, value in dummy_metrics.items():
    print(f"{metric}: {value:.4f}")

Median Price: 5.88

MAE: 1.8245
RMSE: 2.1899
R2: -0.0000


In [ ]:
# ===== C06 - Linear Regression Pipeline =====
numeric_features = [
    "Area",
    "Frontage",
    "Access Road",
    "Floors",
    "Bedrooms",
    "Bathrooms",
]

categorical_features = [
    "House direction",
    "Balcony direction",
    "Legal status",
    "Furniture state",
    "City",
    "District",
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

In [ ]:
# ===== C07 - Linear Regression Evaluation =====

lr_pipeline.fit(
    X_train,
    y_train
)

train_pred = lr_pipeline.predict(
    X_train
)

val_pred = lr_pipeline.predict(
    X_val
)

train_metrics = evaluate(
    y_train,
    train_pred
)

val_metrics = evaluate(
    y_val,
    val_pred
)

print("=== TRAIN ===")
for k, v in train_metrics.items():
    print(f"{k}: {v:.4f}")

print("\n=== VALIDATION ===")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}")

=== TRAIN ===
MAE: 1.2359
RMSE: 1.5849
R2: 0.4875

=== VALIDATION ===
MAE: 1.2635
RMSE: 1.6433
R2: 0.4369


## M08 — Đọc kết quả Linear Regression

### Kết quả

| Dataset | MAE | RMSE | R² |
|----------|----------:|----------:|----------:|
| Dummy Validation | 1.8245 | 2.1899 | ~0 |
| Linear Validation | 1.2635 | 1.6433 | 0.4369 |

### Diễn giải

Linear Regression vượt Dummy trên tất cả metric, cho thấy các feature hiện tại chứa thông tin hữu ích cho việc dự đoán giá nhà.

Khoảng cách giữa train và validation nhỏ (R²: 0.4875 so với 0.4369), chưa xuất hiện dấu hiệu overfitting mạnh.

Tuy nhiên, R² validation vẫn thấp hơn mục tiêu dự án (R² ≥ 0.7), cho thấy mô hình tuyến tính chưa mô tả đầy đủ mối quan hệ giữa đặc trưng bất động sản và giá nhà.

### Quyết định

Tiếp tục huấn luyện Random Forest trên cùng split và cùng feature set để đánh giá lợi ích của mô hình phi tuyến.

Không sử dụng calibration hoặc test trong giai đoạn này.

In [ ]:
# ===== C09 - Random Forest Baseline =====

from sklearn.ensemble import RandomForestRegressor

# Tạo pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

# Train
rf_pipeline.fit(
    X_train,
    y_train,
)

# Predict
rf_train_pred = rf_pipeline.predict(X_train)
rf_val_pred = rf_pipeline.predict(X_val)

# Evaluate
rf_train_metrics = evaluate(
    y_train,
    rf_train_pred,
)

rf_val_metrics = evaluate(
    y_val,
    rf_val_pred,
)

# Print results
print("=== TRAIN ===")
for k, v in rf_train_metrics.items():
    print(f"{k}: {v:.4f}")

print("\n=== VALIDATION ===")
for k, v in rf_val_metrics.items():
    print(f"{k}: {v:.4f}")

=== TRAIN ===
MAE: 0.4031
RMSE: 0.5646
R2: 0.9350

=== VALIDATION ===
MAE: 1.0239
RMSE: 1.3881
R2: 0.5982


In [ ]:
# ===== C10 - Model Comparison =====

import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "Dummy Median",
        "MAE": 1.8245,
        "RMSE": 2.1899,
        "R2": 0.0000,
    },
    {
        "Model": "Linear Regression",
        "MAE": 1.2635,
        "RMSE": 1.6433,
        "R2": 0.4369,
    },
    {
        "Model": "Random Forest",
        "MAE": 1.0239,
        "RMSE": 1.3881,
        "R2": 0.5982,
    },
])

comparison.sort_values(
    by="R2",
    ascending=False,
)



,Model,MAE,RMSE,R2
2,Random Forest,1.0239,1.3881,0.5982
1,Linear Regression,1.2635,1.6433,0.4369
0,Dummy Median,1.8245,2.1899,0.0000


## M11 — Kết luận notebook baseline

### Tóm tắt kết quả

Ba baseline đã được đánh giá trên cùng train/validation split:

- Dummy Median tạo mốc tham chiếu ban đầu.
- Linear Regression cải thiện đáng kể so với Dummy.
- Random Forest đạt kết quả validation tốt nhất hiện tại.

### Điều học được

Kết quả cho thấy các feature hiện tại chứa thông tin hữu ích cho việc dự đoán giá nhà.

Random Forest vượt Linear Regression trên validation, cho thấy bài toán có các quan hệ phi tuyến mà mô hình tuyến tính chưa mô tả tốt.

Tuy nhiên, Random Forest cũng xuất hiện khoảng cách đáng kể giữa train và validation, cần theo dõi khả năng overfitting trong các bước tiếp theo.

### Quyết định

Random Forest được chọn làm baseline mạnh nhất hiện tại.

Notebook tiếp theo sẽ tập trung vào các model và ý tưởng mới (ví dụ: CatBoost, LightGBM, XGBoost, feature engineering hoặc tuning) với mục tiêu cải thiện kết quả validation trước khi thực hiện đánh giá cuối cùng trên test set.